In [ ]:
#| default_exp review

## Reviewing notebooks

Diff and style feedback that focuses on code cells.

Review should focus on the behavior encoded in code cells, not on notebook metadata churn. This notebook keeps that review loop small: run fast.ai style hints when desired, and print nbdev-style code diffs when comparing notebooks.

Review tools draw a line between source changes and notebook noise. `style_check` is useful before publishing exported code because it combines fast.ai style hints with notebook hygiene reports, while `diff_nb` is useful during agent edits because it ignores outputs and metadata unless metadata is the only thing that changed.

### Production contract

Review tools are production core. Validation must fail on missing or stale nbskill metadata, `style_check` must combine capped style output with notebook hygiene diagnostics, and `diff_nb` must show code-cell behavior changes without raw notebook metadata noise.

In [ ]:
import os
from fastcore.nbio import new_nb
from fastcore.test import test_eq
from nbskill.foundation import demo_path, remove_demo_path, write_demo_notebook, write_tool_notebook


def _write_review_notebook(path, cells, stamp=True):
    nb = new_nb(cells)
    if stamp: stamp_notebook_metadata(nb)
    _write_nb(nb, path)
    return nb

In [ ]:
#| export
import ast
import json
import re
import subprocess
from collections import Counter
from contextlib import redirect_stderr, redirect_stdout
from io import StringIO
from pathlib import Path

from chkstyle.core import main as _chkstyle_main
from fastcore.nbio import mk_cell, read_nb
from fastcore.nbio import write_nb as _write_nb
from nbdev.diff import nbs_pair, source_diff
from nbdev.doclinks import nbdev_export as _run_nb_export
from remold import cst, m, cstmap, code

from nbskill.foundation import (
    empty_failure_map, failure_map_path, load_failure_map, cell_class_names, cell_semantic_warnings,
    cell_source, clear_outputs, api_error, api_return, exported_py_path, file_hash, file_line_count,
    cap_text, notebook_paths, parse_code_cell, short_call_name, source_without_directives,
    stamp_notebook_metadata, is_exported_code_cell, none_if_string,
)
from nbskill.parallel import notebook_locks

### Style feedback as a tool

`style_check` wraps the fast.ai style checker as a direct Python API; MCP exposes its common diagnostics through `doctor`.

In [ ]:
#| export
skip_style_paths = "_proc __pycache__ src assets data examples tests archive manage.ipynb".split(" ")
_validation_skip_paths = "_proc __pycache__ src assets examples tests archive manage.ipynb".split(" ")

In [ ]:

#| export
_LARGE_CODE_CELL_LINE_LIMIT = 30
_LARGE_MARKDOWN_CELL_LINE_LIMIT = 20

In [ ]:
#| export
_LARGE_CELL_FUNCTION_LIMIT = 2

In [ ]:
#| export
_LARGE_GENERATED_PY_LINE_LIMIT = 1000

In [ ]:
#| export
def _style_skip_paths(skip_path=None):
    paths = list(skip_style_paths)
    if skip_path is None: return paths
    if isinstance(skip_path, (list, tuple, set)): paths += [str(item) for item in skip_path]
    else: paths.append(str(skip_path))
    return paths

In [ ]:
#| export
def _style_root_is_skipped(path, skip_paths):
    parts = Path("." if path is None else path).expanduser().parts
    return any(part in skip_paths for part in parts)

In [ ]:
#| export
def _explicit_notebook_path(path):
    return Path("." if path is None else path).expanduser().suffix == ".ipynb"

In [ ]:
#| export
def _style_check_argv(path=".", skip_folder_re=None, skip_path=None):
    path = "." if path is None else str(path)
    argv = ["style_check", path]
    if skip_folder_re: argv += ["--skip-path-re", str(skip_folder_re)]
    for item in _style_skip_paths(skip_path): argv += ["--skip-path", item]
    return argv

In [ ]:
#| export
def _top_level_function_count(tree):
    return sum(isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)) for node in tree.body)

In [ ]:
#| export
def _assert_count(tree):
    return sum(isinstance(node, ast.Assert) for node in ast.walk(tree))

In [ ]:
#| export
def _test_function_count(tree):
    return sum(isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)) and node.name.startswith("test_") for node in tree.body)

In [ ]:
#| export
def _public_functions(tree):
    return [
        node for node in tree.body
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef))
        and not node.name.startswith("_")
        and not node.name.startswith("test_")
        and not node.name.startswith("visit_")
        and not node.name.endswith("_tool")
        and node.name not in {"main", "cli"}
    ]

In [ ]:
#| export
def _docstring_line_count(node):
    doc = ast.get_docstring(node, clean=False)
    if doc is None: return 0
    return len(doc.splitlines())

In [ ]:
#| export
def _public_function_docstring_problems(nb_path, cell, tree):
    if not is_exported_code_cell(cell): return []
    problems = []
    for node in _public_functions(tree):
        line_count = _docstring_line_count(node)
        if line_count == 1: continue
        if line_count == 0:
            detail = f"public function {node.name!r} needs a one-line docstring for context summaries"
        else:
            detail = f"public function {node.name!r} has {line_count} docstring lines; keep it to one line for context summaries"
        problems.append(_problem(
            "public-function-docstring", nb_path, cell, detail,
            symbol=node.name, line=getattr(node, "lineno", None), docstring_lines=line_count,
        ))
    return problems

Public notebook APIs stay maintainable when their docs, implementation, examples, and tests travel together. `public_function_literacy_problems` checks exported public functions for that compact contract so maintainers and agents can quickly understand what each function is for, how to call it, and what behavior is protected.

In [ ]:
#| export
def _public_names_in_node(node, public_names):
    names = set()
    for child in ast.walk(node):
        if isinstance(child, ast.Name) and child.id in public_names:
            names.add(child.id)
        elif isinstance(child, ast.Attribute) and child.attr in public_names:
            names.add(child.attr)
    return names

In [ ]:
#| export
def _called_public_names(tree, public_names):
    names = set()
    for node in ast.walk(tree):
        if isinstance(node, ast.Call):
            name = short_call_name(node.func)
            if name in public_names: names.add(name)
    return names

In [ ]:
#| export
def _tested_public_names(tree, public_names):
    names = set()
    test_call_names = {"test_eq", "test_ne", "test_fail"}
    for node in ast.walk(tree):
        if isinstance(node, ast.Assert):
            names.update(_public_names_in_node(node.test, public_names))
        elif isinstance(node, ast.Call) and short_call_name(node.func) in test_call_names:
            for arg in [*node.args, *[kw.value for kw in node.keywords]]:
                names.update(_public_names_in_node(arg, public_names))
    return names

In [ ]:
#| export
def _markdown_explains_function(source, name):
    words = re.findall(r"[A-Za-z][A-Za-z0-9_]*", source)
    return bool(re.search(rf"\b{re.escape(name)}\b", source)) and len(words) >= 5

def _public_function_reference_sets(nb, public_names):
    references = {"markdown": set(), "example": set(), "test": set()}
    for cell in nb.cells:
        source = cell_source(cell)
        if getattr(cell, "cell_type", None) == "markdown":
            references["markdown"].update(
                name for name in public_names if _markdown_explains_function(source, name)
            )
            continue
        tree = parse_code_cell(cell)
        if tree is None or is_exported_code_cell(cell): continue
        tested = _tested_public_names(tree, public_names)
        if tested: references["test"].update(tested)
        elif "test_code" not in cell_class_names(cell):
            references["example"].update(_called_public_names(tree, public_names))
    return references

In [ ]:
#| export
def _public_function_contract_problem(nb_path, cell, node, code, missing, detail="", **kwargs):
    if not detail: detail = f"public function {node.name!r} needs {missing}"
    return _problem(
        code, nb_path, cell, detail,
        symbol=node.name, line=getattr(node, "lineno", None), missing=missing, **kwargs,
    )

In [ ]:
#| export
def _public_function_literacy_problems_for_nb(nb_path, nb):
    public_functions = []
    for cell in nb.cells:
        tree = parse_code_cell(cell)
        if tree is None or not is_exported_code_cell(cell): continue
        public_functions.extend((cell, node) for node in _public_functions(tree))
    public_names = {node.name for _, node in public_functions}
    references = _public_function_reference_sets(nb, public_names)
    problems = []
    for cell, node in public_functions:
        line_count = _docstring_line_count(node)
        if line_count == 0:
            detail = f"public function {node.name!r} needs a one-line docstring for context summaries"
            problems.append(_public_function_contract_problem(nb_path, cell, node, "public-function-docstring", "one-line docstring", detail, docstring_lines=line_count))
        elif line_count != 1:
            detail = f"public function {node.name!r} has {line_count} docstring lines; keep it to one line for context summaries"
            problems.append(_public_function_contract_problem(nb_path, cell, node, "public-function-docstring", "one-line docstring", detail, docstring_lines=line_count))
        if node.name not in references["markdown"]:
            detail = f"public function {node.name!r}: add a short Markdown rationale cell mentioning the function after the exported code"
            problems.append(_public_function_contract_problem(nb_path, cell, node, "public-function-markdown", "short Markdown rationale cell", detail))
        if node.name not in references["example"]:
            detail = f"public function {node.name!r}: add a small example cell that calls the function"
            problems.append(_public_function_contract_problem(nb_path, cell, node, "public-function-example", "small example cell that calls the function", detail))
        if node.name not in references["test"]:
            detail = f"public function {node.name!r}: add a focused test cell that asserts the behavior"
            problems.append(_public_function_contract_problem(nb_path, cell, node, "public-function-test", "focused test cell that asserts the behavior", detail))
    return problems

In [ ]:
#| export
def public_function_literacy_problems(path="."):
    "Return docs/example/test contract warnings for exported public notebook functions."
    problems = []
    for nb_path in notebook_paths(path):
        try: nb = read_nb(nb_path)
        except FileNotFoundError: continue
        problems.extend(_public_function_literacy_problems_for_nb(nb_path, nb))
    return problems

In [ ]:
#| export
def _import_keys(tree):
    keys = []
    for node in tree.body:
        if isinstance(node, ast.Import):
            for alias in node.names:
                local = alias.asname or alias.name.split(".", 1)[0]
                keys.append(f"import {alias.name} as {local}")
        elif isinstance(node, ast.ImportFrom):
            module = "." * node.level + (node.module or "")
            for alias in node.names:
                local = alias.asname or alias.name
                keys.append(f"from {module} import {alias.name} as {local}")
    return keys

In [ ]:
#| export
def _cell_content_line_count(cell):
    source = cell_source(cell)
    if getattr(cell, "cell_type", None) == "code": source = source_without_directives(source)
    return len([line for line in source.splitlines() if line.strip()])

In [ ]:
#| export
def _problem(code, path, cell=None, detail="", severity="warning", source="nbskill", **kwargs):
    problem = dict(code=code,path=str(path),severity=severity,source=source,detail=detail)
    if cell is not None: problem["cell_id"] = getattr(cell, "id", "")
    problem.update({key: value for key, value in kwargs.items() if value is not None})
    return problem

In [ ]:
#| export
def _single_top_level_function_cell(tree):
    return len(tree.body) == 1 and isinstance(tree.body[0], (ast.FunctionDef, ast.AsyncFunctionDef))

In [ ]:
#| export
def _large_cell_problems(nb_path, cell, function_limit=_LARGE_CELL_FUNCTION_LIMIT):
    problems = []
    line_count = _cell_content_line_count(cell)
    cell_type = getattr(cell, "cell_type", "cell")
    tree = parse_code_cell(cell)
    code_cell = cell_type == "code" and tree is not None
    line_limit = _LARGE_CODE_CELL_LINE_LIMIT if code_cell else _LARGE_MARKDOWN_CELL_LINE_LIMIT
    allow_long_cell = code_cell and _single_top_level_function_cell(tree)
    if line_count > line_limit and not allow_long_cell:
        detail = f"{line_count} {cell_type} content lines; keep one idea per cell; split this into markdown rationale, one focused implementation cell, example, and test when it exports behavior"
        problems.append(_problem("large-cell", nb_path, cell, detail, line_count=line_count))
    if tree is None: return problems
    function_count = _top_level_function_count(tree)
    if function_count > function_limit:
        detail = f"{function_count} top-level functions; split into one-idea cells; try edit_notebook op=\"explode_cells\" for this cell"
        problems.append(_problem("large-cell", nb_path, cell, detail, function_count=function_count))
    return problems

In [ ]:
#| export
def _generated_py_size_problem(nb_path, nb, line_limit=_LARGE_GENERATED_PY_LINE_LIMIT):
    py_path = exported_py_path(nb_path, nb)
    if py_path is None or not py_path.exists(): return None
    line_count = file_line_count(py_path)
    if line_count <= line_limit: return None
    detail = f"{line_count} generated Python lines; split this notebook/module with the split tool"
    return _problem("large-generated-py", nb_path, detail=detail, exported_py_path=str(py_path), line_count=line_count)

In [ ]:
#| export
def _format_problem(problem):
    cell = f" id={problem['cell_id']}" if problem.get("cell_id") else ""
    line = f" line={problem['line']}" if problem.get("line") else ""
    fields = []
    for key in (
        "scope", "symbol", "missing", "import_key", "cells", "semantic_types", "confidence", "hint",
        "exported_py_path", "line_count", "function_count", "docstring_lines",
        "regex", "note", "match", "candidate", "target_path", "target_cell_id",
        "score", "gap", "reason",
    ):
        if key in problem:
            value = problem[key]
            if isinstance(value, list): value = ", ".join(map(str, value))
            quoted = {
                "symbol", "missing", "import_key", "exported_py_path", "regex",
                "note", "match", "candidate", "target_path", "target_cell_id",
                "reason",
            }
            fields.append(f"{key}={value!r}" if key in quoted else f"{key}={value}")
    suffix = " ".join(fields + ([problem.get("detail", "")] if problem.get("detail") else []))
    return f"- {problem['code']}: {problem['path']}{cell}{line} {suffix}".rstrip()

Notebook craft is a small subset of style diagnostics that should steer agents toward literate, reviewable notebook changes.

In [ ]:
#| export
_CRAFT_PROBLEM_CODES = {
    "large-cell", "large-generated-py",
    "public-function-docstring", "public-function-markdown", "public-function-example", "public-function-test",
}
_STORY_PROBLEM_CODES = {"public-function-markdown", "public-function-example", "public-function-test"}

In [ ]:
#| export
def _craft_diagnostics(diagnostics):
    return [item for item in diagnostics if item.get("code") in _CRAFT_PROBLEM_CODES]

In [ ]:
#| export
def _craft_summary(diagnostics):
    craft = _craft_diagnostics(diagnostics)
    return dict(
        craft_problem_count=len(craft),
        large_cell_count=sum(1 for item in craft if item.get("code") == "large-cell"),
        large_generated_py_count=sum(1 for item in craft if item.get("code") == "large-generated-py"),
        public_function_story_problem_count=sum(1 for item in craft if item.get("code") in _STORY_PROBLEM_CODES),
        public_function_docstring_problem_count=sum(1 for item in craft if item.get("code") == "public-function-docstring"),
    )

In [ ]:
#| export
def _format_notebook_craft_report(diagnostics):
    craft = _craft_diagnostics(diagnostics)
    if not craft: return "Notebook craft: no large-cell or story diagnostics found."
    return "\n".join(["Notebook craft:", *[_format_problem(problem) for problem in craft]])

In [ ]:
_literacy_preview = public_function_literacy_problems("nbs/04_review.ipynb")[:2]
_literacy_preview

[{'code': 'public-function-markdown',
  'path': '04_review.ipynb',
  'severity': 'warning',
  'source': 'nbskill',
  'detail': "public function 'notebook_size_problems' needs Markdown docs cell mentioning the function",
  'cell_id': '93269841',
  'symbol': 'notebook_size_problems',
  'line': 1,
  'missing': 'Markdown docs cell mentioning the function'},
 {'code': 'public-function-example',
  'path': '04_review.ipynb',
  'severity': 'warning',
  'source': 'nbskill',
  'detail': "public function 'notebook_size_problems' needs example cell that calls the function",
  'cell_id': '93269841',
  'symbol': 'notebook_size_problems',
  'line': 1,
  'missing': 'example cell that calls the function'}]

In [ ]:
assert {
    "public-function-docstring", "public-function-markdown",
    "public-function-example", "public-function-test",
} <= {problem["code"] for problem in public_function_literacy_problems("nbs/04_review.ipynb")}

In [ ]:
#| export
def _nbskill_info(obj):
    meta = getattr(obj, "metadata", {}) or {}
    return meta.get("nbskill") if isinstance(meta, dict) else None

In [ ]:
#| export
def _validation_problem(code, path, cell=None, detail="", **kwargs):
    return _problem(code, path, cell, detail=detail, severity="error", source="nbskill-validation", **kwargs)

In [ ]:
#| export
def _cell_metadata_validation_problems(nb_path, cell):
    info = _nbskill_info(cell)
    problems = []
    if not isinstance(info, dict):
        problems.append(_validation_problem("missing-cell-nbskill-metadata", nb_path, cell))
    else:
        expected_type = getattr(cell, "cell_type", None)
        if "cell_type" not in info:
            problems.append(_validation_problem("missing-cell-type", nb_path, cell))
        elif info.get("cell_type") != expected_type:
            problems.append(_validation_problem("cell-type-mismatch", nb_path, cell, f"expected {expected_type!r}, stored {info.get('cell_type')!r}"))
        semantic_types = info.get("semantic_types")
        if not isinstance(semantic_types, list) or not semantic_types:
            problems.append(_validation_problem("missing-cell-semantic-types", nb_path, cell))
    syntax_problem = _cell_python_syntax_problem(nb_path, cell)
    if syntax_problem: problems.append(syntax_problem)
    return problems

In [ ]:
#| exporti
def _cell_python_syntax_problem(nb_path, cell):
    if getattr(cell, "cell_type", None) != "code": return None
    try: ast.parse(source_without_directives(cell_source(cell)))
    except SyntaxError as exc:
        source_line = exc.text.strip() if exc.text else ""
        detail = f"{exc.msg}: {source_line}" if source_line else exc.msg
        return _validation_problem(
            "invalid-python-cell", nb_path, cell, detail,
            line=exc.lineno, column=exc.offset,
        )

In [ ]:
#| export
def _notebook_export_hash_problems(nb_path, nb):
    py_path = exported_py_path(nb_path, nb)
    if py_path is None: return []
    info = _nbskill_info(nb)
    if not py_path.exists():
        return [_validation_problem("exported-py-missing", nb_path, detail=f"missing {py_path}", exported_py_path=str(py_path))]
    if not isinstance(info, dict) or not info.get("exported_py_hash"):
        return [_validation_problem("missing-exported-py-hash", nb_path, exported_py_path=str(py_path))]
    actual = file_hash(py_path)
    stored = info.get("exported_py_hash")
    if stored != actual:
        detail = f"expected {actual[:12]}, stored {str(stored)[:12]}"
        return [_validation_problem("exported-py-hash-mismatch", nb_path, detail=detail, exported_py_path=str(py_path))]
    return []

In [ ]:
#| export
def notebook_validation_problems(path="."):
    "Return notebook metadata and Python syntax validation problems."
    problems = []
    for nb_path in notebook_paths(path):
        if _style_root_is_skipped(nb_path, _validation_skip_paths): continue
        try: nb = read_nb(nb_path)
        except FileNotFoundError: continue
        problems.extend(_notebook_export_hash_problems(nb_path, nb))
        for cell in nb.cells: problems.extend(_cell_metadata_validation_problems(nb_path, cell))
    return problems

In [ ]:
#| export
def _notebook_size_problems_for_nb(nb_path, nb):
    problems = []
    for cell in nb.cells: problems.extend(_large_cell_problems(nb_path, cell))
    generated_problem = _generated_py_size_problem(nb_path, nb)
    if generated_problem: problems.append(generated_problem)
    return problems

In [ ]:
#| export
def notebook_size_problems(path="."):
    "Return size warnings for notebook cells and generated Python files."
    problems = []
    explicit = _explicit_notebook_path(path)
    for nb_path in notebook_paths(path):
        if not explicit and _style_root_is_skipped(nb_path, skip_style_paths): continue
        try: nb = read_nb(nb_path)
        except FileNotFoundError: continue
        problems.extend(_notebook_size_problems_for_nb(nb_path, nb))
    return problems

In [ ]:
#| export
def _style_path_is_skipped(path, skip_paths, skip_folder_re=None):
    if _style_root_is_skipped(path, skip_paths): return True
    return bool(skip_folder_re and re.search(str(skip_folder_re), str(path)))

In [ ]:
#| export
def _notebook_style_problems(path=".", skip_folder_re=None, skip_path=None):
    skip_paths = _style_skip_paths(skip_path)
    explicit = _explicit_notebook_path(path)
    problems = notebook_validation_problems(path)
    duplicate_imports = {}
    for nb_path in notebook_paths(path):
        if not explicit and _style_path_is_skipped(nb_path, skip_paths, skip_folder_re): continue
        try:
            nb = read_nb(nb_path)
        except FileNotFoundError:
            continue
        problems.extend(_notebook_size_problems_for_nb(nb_path, nb))
        problems.extend(_public_function_literacy_problems_for_nb(nb_path, nb))
        imports_by_scope = {"exported": {}, "internal": {}}
        for cell in nb.cells:
            classes = cell_class_names(cell)
            for code in cell_semantic_warnings(cell):
                problems.append(_problem(code, nb_path, cell, semantic_types=list(classes)))
            tree = parse_code_cell(cell)
            if tree is None: continue
            if "test_code" in classes:
                problem_count = _assert_count(tree) + _test_function_count(tree)
                if problem_count > 3:
                    problems.append(_problem("multi-problem-test", nb_path, cell, f"{problem_count} asserts/test functions; split into one-problem-at-a-time cells"))
            scope = "exported" if is_exported_code_cell(cell) else "internal"
            for key in _import_keys(tree):
                imports_by_scope[scope].setdefault(key, []).append(getattr(cell, "id", ""))
        for scope, imports in imports_by_scope.items():
            for key, ids in imports.items():
                if len(ids) > 1:
                    duplicate_imports.setdefault(str(nb_path), []).append((scope, key, ids))
    for nb_path, items in duplicate_imports.items():
        for scope, key, ids in items:
            problems.append(_problem("duplicate-import", nb_path, scope=scope, import_key=key, cells=ids))
    if notebook_paths(path):
        from nbskill.graph import notebook_order_problems
        problems.extend(
            problem for problem in notebook_order_problems(path)
            if explicit or not _style_path_is_skipped(problem["path"], skip_paths, skip_folder_re)
        )
    return problems

In [ ]:
#| export
def _notebook_style_problem_lines(path=".", skip_folder_re=None, skip_path=None):
    problems = _notebook_style_problems(path, skip_folder_re=skip_folder_re, skip_path=skip_path)
    return [_format_problem(problem) for problem in problems]

In [ ]:
#| export
def _format_notebook_style_report(path=".", skip_folder_re=None, skip_path=None):
    problems = _notebook_style_problems(path, skip_folder_re=skip_folder_re, skip_path=skip_path)
    if not problems: return "Notebook style report: no notebook hygiene problems found."
    craft_text = _format_notebook_craft_report(problems)
    lines = [_format_problem(problem) for problem in problems]
    return "\n".join(["Notebook style report:", craft_text, *lines])

In [ ]:
#| export
def _format_count_group(title, counts):
    if not counts: return [f"{title}: none"]
    ordered = sorted(counts.items(), key=lambda item: (-item[1], item[0]))
    return [f"{title}: " + ", ".join(f"{tool}={count}" for tool, count in ordered)]

In [ ]:
#| export
def _format_failure_event(event):
    kind = event.get("kind", "event")
    tool = event.get("tool", "unknown")
    path = event.get("path")
    detail = event.get("summary") or event.get("error") or ",".join(event.get("reasons", []))
    location = f" path={path}" if path else ""
    return f"- {kind}: {tool}{location} {detail}".rstrip()

In [ ]:
#| export
def _global_usage_summary_data():
    path = failure_map_path()
    data = load_failure_map(path) if path.exists() else empty_failure_map()
    problems = [event for event in data.get("events", []) if event.get("kind") in {"failure", "friction"}][-5:]
    return {
        "path": str(path),
        "exists": path.exists(),
        "counts": data.get("counts", {}),
        "recent_problems": problems,
    }

In [ ]:
#| export
def _format_global_usage_summary(data=None):
    data = data or _global_usage_summary_data()
    if not data["exists"]: return f"Global nbskill usage: no records at {data['path']}"
    counts = data.get("counts", {})
    lines = [f"Global nbskill usage: {data['path']}"]
    lines += _format_count_group("usage", counts.get("usage", {}))
    lines += _format_count_group("failures", counts.get("failures", {}))
    lines += _format_count_group("friction", counts.get("friction", {}))
    problems = data.get("recent_problems", [])
    if problems:
        lines.append("recent problems:")
        lines.extend(_format_failure_event(event) for event in problems)
    else:
        lines.append("recent problems: none")
    return "\n".join(lines)

In [ ]:
#| export
def reset_global_usage_summary():
    path = failure_map_path()
    try: path.unlink()
    except FileNotFoundError: pass
    except OSError: path.write_text(json.dumps(empty_failure_map(), indent=2, sort_keys=True), encoding="utf-8")

In [ ]:
#| export
_CHKSTYLE_RE = re.compile(r"^# (?P<path>.*?):cell\[(?P<cell>[^\]]+)\]:(?P<line>\d+): (?P<detail>.*)$")

In [ ]:
#| export
def _chkstyle_diagnostics(text, max_diagnostics=200):
    diagnostics = []
    for line in (text or "").splitlines():
        match = _CHKSTYLE_RE.match(line)
        if not match: continue
        detail = match.group("detail")
        hint = None
        if "(hint:" in detail:
            detail, hint = detail.split("(hint:", 1)
            hint = hint.rstrip(")").strip()
        diagnostics.append({
            "source": "chkstyle",
            "code": detail.strip().split(" (", 1)[0],
            "path": match.group("path"),
            "cell_id": match.group("cell"),
            "line": int(match.group("line")),
            "severity": "hint",
            "detail": detail.strip(),
            "hint": hint,
        })
        if max_diagnostics and len(diagnostics) >= max_diagnostics: break
    return diagnostics

In [ ]:
#| export
def _problem_chart(diagnostics):
    def counts(key):
        return dict(sorted(Counter(str(item.get(key, "")) for item in diagnostics if item.get(key)).items()))
    return {
        "by_code": counts("code"),
        "by_severity": counts("severity"),
        "by_path": counts("path"),
        "by_source": counts("source"),
    }

In [ ]:
#| export
def _fix_suggestions(diagnostics):
    fixes = []
    for item in diagnostics:
        if item.get("code") == "duplicate-import":
            fixes.append({
                "code": "duplicate-import",
                "path": item.get("path"),
                "cells": item.get("cells", []),
                "description": f"Remove repeated import {item.get('import_key')} from later cells after checking scope.",
                "automatic": False,
            })
    return fixes

In [ ]:
#| export
def _merge_line_to_previous(lines, idx, sep=""):
    if idx <= 0 or idx >= len(lines): return False
    lines[idx - 1] = f"{lines[idx - 1].rstrip()}{sep}{lines[idx].strip()}"
    del lines[idx]
    return True

In [ ]:
#| export
def _add_fix_count(counts, code, value=1):
    if value: counts[code] = counts.get(code, 0) + value

In [ ]:
#| export
def _split_directive_lines(source):
    lines = str(source or "").splitlines()
    idx = 0
    while idx < len(lines) and lines[idx].lstrip().startswith("#|"): idx += 1
    return lines[:idx], lines[idx:]

In [ ]:
#| export
def _source_ast_dump(source):
    try: tree = ast.parse(source_without_directives(source))
    except SyntaxError: return None
    return ast.dump(tree, include_attributes=False)

In [ ]:
#| export
def _same_ast(before, after):
    before_dump, after_dump = _source_ast_dump(before), _source_ast_dump(after)
    return before_dump is not None and before_dump == after_dump

In [ ]:
#| export
def _safe_python_source(source):
    _, lines = _split_directive_lines(source)
    if not "\n".join(lines).strip(): return False
    for line in lines:
        stripped = line.lstrip()
        if stripped and (stripped.startswith(("%", "!", "?")) or stripped.endswith("?")): return False
    return True

In [ ]:
#| export
def _source_transform_result(source, fixes):
    fixes = {key: value for key, value in fixes.items() if value}
    return {"source": source, "fixes": fixes, "changed": bool(fixes)}

In [ ]:
#| export
def _apply_source_transform(source, code, fn, ast_equal=True):
    try: result = fn(source)
    except Exception: return source, {}
    fixed, count = result if isinstance(result, tuple) else (result, int(result != source))
    if not fixed or fixed == source: return source, {}
    fixed = fixed.rstrip("\n") + "\n"
    if ast_equal and not _same_ast(source, fixed): return source, {}
    return fixed, {code: count or 1}

In [ ]:
#| export
def _string_literal_lines(tree):
    lines = set()
    for node in ast.walk(tree):
        if not (isinstance(node, ast.Constant) and isinstance(node.value, str)): continue
        start = getattr(node, "lineno", 0)
        end = getattr(node, "end_lineno", start)
        lines.update(range(start, end + 1))
    return lines

In [ ]:
#| export
def _function_body_lines(tree):
    lines = set()
    for node in ast.walk(tree):
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)):
            lines.update(range(node.lineno + 1, getattr(node, "end_lineno", node.lineno) + 1))
    return lines

In [ ]:
#| export
def _remove_function_blank_lines(lines):
    try: tree = ast.parse("\n".join(lines))
    except SyntaxError: return 0
    body_lines = _function_body_lines(tree)
    protected = _string_literal_lines(tree)
    removed = 0
    for idx in range(len(lines) - 1, -1, -1):
        line_no = idx + 1
        if line_no in body_lines and line_no not in protected and not lines[idx].strip():
            del lines[idx]
            removed += 1
    return removed

In [ ]:
#| export
_SIMPLE_CST_BODY_TYPES = (cst.Assign, cst.AnnAssign, cst.AugAssign, cst.Expr, cst.Return, cst.Raise, cst.Pass, cst.Break, cst.Continue, cst.Del)


def _compact_simple_ifs_with_remold(source, max_len=180):
    changed = 0

    def fix(node, _):
        nonlocal changed
        if node.orelse or not isinstance(node.body, cst.IndentedBlock): return None
        if node.body.header.comment or len(node.body.body) != 1: return None
        stmt = node.body.body[0]
        if not isinstance(stmt, cst.SimpleStatementLine) or stmt.leading_lines: return None
        if len(stmt.body) != 1 or not isinstance(stmt.body[0], _SIMPLE_CST_BODY_TYPES): return None
        body = code(stmt).strip()
        if "\n" in body: return None
        replacement = f"if {code(node.test).strip()}: {body}"
        if len(replacement) > max_len: return None
        changed += 1
        return replacement

    return cstmap(m.If(), fix)(source), changed

In [ ]:
#| export
def _merge_paren_only_lines(lines):
    merged = 0
    idx = 1
    while idx < len(lines):
        if lines[idx].strip() in {")", "]", "}"} and lines[idx - 1].strip():
            if _merge_line_to_previous(lines, idx):
                merged += 1
                continue
        idx += 1
    return merged

In [ ]:
#| export
def _normalize_code_lines(lines):
    counts = {}
    stripped = [line.rstrip() for line in lines]
    _add_fix_count(counts, "trailing-whitespace", sum(old != new for old, new in zip(lines, stripped)))
    lines[:] = stripped
    compacted, blank_run, removed = [], 0, 0
    for line in lines:
        if line.strip():
            blank_run = 0
            compacted.append(line)
        else:
            blank_run += 1
            if blank_run <= 2: compacted.append("")
            else: removed += 1
    lines[:] = compacted
    _add_fix_count(counts, "excess-blank-lines", removed)
    return counts

In [ ]:
#| export
def _fix_code_cell_source(source):
    original = str(source or "")
    if not _safe_python_source(original): return {"source": original, "fixes": {}, "changed": False}
    before_dump = _source_ast_dump(original)
    if before_dump is None: return {"source": original, "fixes": {}, "changed": False}
    directives, lines = _split_directive_lines(original)
    counts = _normalize_code_lines(lines)
    _add_fix_count(counts, "function-blank-line", _remove_function_blank_lines(lines))
    _add_fix_count(counts, "paren-only-line", _merge_paren_only_lines(lines))
    if not original.endswith("\n"): _add_fix_count(counts, "final-newline")
    new_source = "\n".join([*directives, *lines]).rstrip("\n") + "\n"
    new_source, compact = _apply_source_transform(new_source, "compact-if", _compact_simple_ifs_with_remold)
    counts.update(compact)
    if compact: counts["single-line-if"] = compact["compact-if"]
    if new_source == original: return {"source": original, "fixes": {}, "changed": False}
    if _source_ast_dump(new_source) != before_dump: return {"source": original, "fixes": {}, "changed": False}
    return _source_transform_result(new_source, counts)

In [ ]:
#| export
def _cst_split_definition(stmt):
    return isinstance(stmt, (cst.FunctionDef, cst.ClassDef)) and not stmt.decorators

In [ ]:
#| export
def _cst_definition_split_sources(source):
    if not _safe_python_source(source): return []
    try: mod = cst.parse_module(source)
    except Exception: return []
    body = list(mod.body)
    if len(body) < 2 or not all(_cst_split_definition(stmt) for stmt in body): return []
    header = "".join(code(line) for line in mod.header)
    chunks = []
    for idx, stmt in enumerate(body):
        chunk = (header if idx == 0 else "") + code(stmt)
        chunk = chunk.strip("\n") + "\n"
        if not _safe_python_source(chunk) or _source_ast_dump(chunk) is None: return []
        chunks.append(chunk)
    return chunks if _same_ast(source, "".join(chunks)) else []

In [ ]:
#| export
_AUTOFIX_LABELS = {
    "trailing-whitespace": "removed trailing whitespace",
    "excess-blank-lines": "collapsed excessive blank lines",
    "function-blank-line": "removed function-body blank lines",
    "compact-if": "compacted simple if statements",
    "single-line-if": "merged single-line if statements",
    "paren-only-line": "merged parenthesis-only lines",
    "final-newline": "added final newline",
    "split-definitions": "split pure definition cell",
    "preserve-directives": "preserved nbdev directives while splitting",
    "split-functions": "split multi-function cell"}


def _autofix_record(path, cell_id, code, count=1):
    record = {"path": str(path), "cell_id": str(cell_id), "code": code, "count": count}
    record["description"] = f"{_AUTOFIX_LABELS.get(code, code)} ({count})"
    return record

In [ ]:
#| export
def notebook_autofix(path=".", dry_run=False):
    records = []
    for nb_path in notebook_paths(path):
        with notebook_locks(nb_path):
            nb = read_nb(nb_path)
            changed, idx = False, 0
            while idx < len(nb.cells):
                cell = nb.cells[idx]
                if getattr(cell, "cell_type", None) != "code":
                    idx += 1
                    continue
                source = cell_source(cell)
                fixed = _fix_code_cell_source(source)
                if fixed["changed"]:
                    source = fixed["source"]
                    records.extend(_autofix_record(nb_path, cell.id, code, count) for code, count in fixed["fixes"].items())
                    if not dry_run:
                        cell.source = source
                        clear_outputs(cell)
                    changed = True
                split_sources = _cst_definition_split_sources(source)
                if split_sources:
                    records.append(_autofix_record(nb_path, cell.id, "split-definitions", len(split_sources)))
                    records.append(_autofix_record(nb_path, cell.id, "split-functions", len(split_sources)))
                    if source.lstrip().startswith("#|"): records.append(_autofix_record(nb_path, cell.id, "preserve-directives"))
                    if not dry_run:
                        new_cells = [mk_cell(chunk, cell_type="code") for chunk in split_sources]
                        new_cells[0].id = cell.id
                        for new_cell in new_cells: clear_outputs(new_cell)
                        nb.cells[idx:idx + 1] = new_cells
                    changed = True
                    idx += len(split_sources)
                else:
                    idx += 1
            if changed and not dry_run:
                stamp_notebook_metadata(nb)
                _write_nb(nb, nb_path)
                if exported_py_path(nb_path, nb) is not None: _run_nb_export(path=str(nb_path))
    return records

In [ ]:
#| hide
lines = ["items = [", "    1,", "    ]"]
assert _merge_line_to_previous(lines, 2)
assert lines == ["items = [", "    1,]"]

body_lines = ["if ready:", "    return 1"]
assert _merge_line_to_previous(body_lines, 1, sep=" ")
assert body_lines == ["if ready: return 1"]

In [ ]:
#| hide
blank_source = "\n".join([
    "def keep_literals():",
    "    text = '''line",
    "",
    "literal'''",
    "",
    "    return text",
    "",
    "value = 1",
]) + "\n"
blank_fixed = _fix_code_cell_source(blank_source)
assert blank_fixed["changed"]
assert blank_fixed["fixes"] == {"function-blank-line": 1}
assert "line\n\nliteral" in blank_fixed["source"]
assert "literal'''\n    return text" in blank_fixed["source"]
assert "return text\n\nvalue = 1" in blank_fixed["source"]

In [ ]:
#| hide
short_if = "def pick(flag):\n    # keep me\n    if flag:\n        return 1\n"
short_fixed = _fix_code_cell_source(short_if)
assert short_fixed["changed"]
assert short_fixed["fixes"]["compact-if"] == 1
assert short_fixed["fixes"]["single-line-if"] == 1
assert "# keep me\n    if flag: return 1" in short_fixed["source"]
assert _same_ast(short_if, short_fixed["source"])

long_body = "x" * 170
long_if = f"def pick(flag):\n    if flag:\n        value = '{long_body}'\n"
assert not _fix_code_cell_source(long_if)["changed"]

multi_if = "def pick(flag):\n    if flag:\n        print(flag)\n        return 1\n"
assert not _fix_code_cell_source(multi_if)["changed"]

magic_source = "%time value = 1\n"
assert not _fix_code_cell_source(magic_source)["changed"]

In [ ]:
#| hide
paren_source = "\n".join([
    "def values():",
    "    items = [",
    "        1,",
    "        ]",
    "    return items",
]) + "\n"
paren_fixed = _fix_code_cell_source(paren_source)
assert paren_fixed["changed"]
assert paren_fixed["fixes"] == {"paren-only-line": 1}
assert "1,]" in paren_fixed["source"]
assert _source_ast_dump(paren_fixed["source"]) == _source_ast_dump(paren_source)

In [ ]:
#| hide
split_source = "\n".join([
    "#| export",
    "def first():",
    "    return 1",
    "",
    "",
    "class Second:",
    "    pass",
]) + "\n"
split = _cst_definition_split_sources(split_source)
assert split == ["#| export\ndef first():\n    return 1\n", "class Second:\n    pass\n"]
assert _same_ast(split_source, "".join(split))

mixed_import = "import os\n\ndef first():\n    return os.name\n\ndef second():\n    return 2\n"
assert _cst_definition_split_sources(mixed_import) == []
mixed_constant = "VALUE = 1\n\ndef first():\n    return VALUE\n\ndef second():\n    return 2\n"
assert _cst_definition_split_sources(mixed_constant) == []
decorated = "@cache\ndef first():\n    return 1\n\ndef second():\n    return 2\n"
assert _cst_definition_split_sources(decorated) == []
assert _cst_definition_split_sources("?name\n") == []
assert _cst_definition_split_sources("name?\n") == []

In [ ]:
#| hide
from contextlib import nullcontext
style_source = "\n".join([
    "#| export",
    "def first(flag):",
    "",
    "    if flag:",
    "        return 1",
    "",
    "def second():",
    "    return 2",
]) + "\n"

class _AutofixDemoCell:
    cell_type = "code"; id = "remold-style"; source = style_source

demo_nb = type("DemoNb", (), {"cells": [_AutofixDemoCell()]})()
autofix_globals = notebook_autofix.__globals__
old_notebook_paths = autofix_globals["notebook_paths"]
old_read_nb = autofix_globals["read_nb"]
old_notebook_locks = autofix_globals["notebook_locks"]
try:
    autofix_globals["notebook_paths"] = lambda path: [Path("demo.ipynb")]
    autofix_globals["read_nb"] = lambda path: demo_nb
    autofix_globals["notebook_locks"] = lambda path: nullcontext()
    records = notebook_autofix("demo.ipynb", dry_run=True)
    assert {"compact-if", "split-definitions", "preserve-directives"} <= {record["code"] for record in records}
    assert demo_nb.cells[0].source == style_source
finally:
    autofix_globals["notebook_paths"] = old_notebook_paths
    autofix_globals["read_nb"] = old_read_nb
    autofix_globals["notebook_locks"] = old_notebook_locks

In [ ]:
#| export
def style_report(
    path: str = ".",  # File or folder to check
    chkstyle: dict | None = None,  # Captured chkstyle result to include
    max_output_chars: int = 12000,  # Maximum raw chkstyle text to keep in report
    max_diagnostics: int = 200,  # Maximum parsed chkstyle diagnostics
    changed_only: bool = False,  # Report only diagnostics for changed code cells
    ref_a: str | None = "HEAD",  # First git ref for changed_only filtering
    ref_b: str | None = None,  # Second git ref; defaults to working tree
    skip_folder_re: str | None = None,  # Regex for notebook paths to skip
    skip_path: str | None = None,  # Folder name/path to skip
):
    "Return structured style diagnostics, problem chart, and global nbskill usage data."
    notebook_problems = _notebook_style_problems(path, skip_folder_re=skip_folder_re, skip_path=skip_path)
    changed_cell_ids = _changed_code_cell_ids(path, ref_a=ref_a, ref_b=ref_b) if changed_only else None
    if changed_cell_ids is not None and notebook_paths(path):
        skip_paths = _style_skip_paths(skip_path)
        explicit = _explicit_notebook_path(path)
        from nbskill.graph import notebook_advice_problems
        advice_problems = [
            problem for problem in notebook_advice_problems(path)
            if problem.get("cell_id") in changed_cell_ids
            and (explicit or not _style_path_is_skipped(problem.get("path", ""), skip_paths, skip_folder_re))
        ]
        notebook_problems = [
            item for item in notebook_problems if item.get("cell_id") in changed_cell_ids
        ] + advice_problems
    usage = _global_usage_summary_data()
    chkstyle = chkstyle or {"status": 0, "output": ""}
    capped = cap_text(chkstyle.get("output", ""), max_output_chars=max_output_chars)
    diagnostics = _chkstyle_diagnostics(chkstyle.get("output", ""), max_diagnostics=max_diagnostics) + notebook_problems
    if changed_cell_ids is not None:
        diagnostics = [item for item in diagnostics if item.get("cell_id") in changed_cell_ids]
    notebook_text = _format_notebook_style_report(path, skip_folder_re=skip_folder_re, skip_path=skip_path)
    craft_text = _format_notebook_craft_report(diagnostics)
    usage_text = _format_global_usage_summary(usage)
    chkstyle_text = capped["text"].strip()
    if changed_cell_ids is not None: text = _format_style_delta_report(path, diagnostics, changed_cell_ids, usage_text)
    else: text = "\n\n".join(chunk for chunk in [chkstyle_text, craft_text, notebook_text, usage_text] if chunk)
    summary = dict(
        notebook_problem_count=len(notebook_problems),
        chkstyle_problem_count=len([item for item in diagnostics if item.get("source") == "chkstyle"]),
        diagnostic_count=len(diagnostics),
        recent_problem_count=len(usage.get("recent_problems", [])),
        output_truncated=capped["truncated"],
        output_chars=capped["chars"],
        omitted_chars=capped["omitted_chars"],
        changed_only=changed_only,
        changed_cell_count=len(changed_cell_ids or []),
        **_craft_summary(diagnostics))
    return dict(
        path=str(path),
        summary=summary,
        diagnostics=diagnostics[:max_diagnostics] if max_diagnostics else diagnostics,
        problem_chart=_problem_chart(diagnostics),
        notebook_problems=notebook_problems,
        global_usage=usage,
        chkstyle={"status": chkstyle.get("status", 0), **capped},
        fixes=_fix_suggestions(diagnostics),
        text=text)

In [ ]:
with write_tool_notebook("04_review_style_report.ipynb") as path:
    report = style_report(path)
    test_eq({"summary", "notebook_problems"} <= set(report), True)
    test_eq(isinstance(report["notebook_problems"], list), True)
    test_eq("Global nbskill usage:" in report["text"], True)

The structured report is meant for tools as much as people. `summary` gives a compact count of problems, `problem_chart` groups diagnostics by source and code, and `text` is the human-readable report.

In [ ]:
sample_chkstyle = {"status": 1, "output": "# demo.ipynb:cell[abc123]:2: Missing whitespace (hint: add a blank line)"}
with write_tool_notebook("04_review_sample_report.ipynb") as path:
    sample_report = style_report(path, chkstyle=sample_chkstyle, max_output_chars=200, max_diagnostics=5)
    print(sample_report["summary"])
    print(sample_report["problem_chart"]["by_source"])
    print(sample_report["diagnostics"][0])

In [ ]:
#| export
def run_style_check(path=".", skip_folder_re=None, skip_path=None, strict=False, max_output_chars=None):
    "Run chkstyle with nbskill's default skip paths and capped output."
    skip_paths = _style_skip_paths(skip_path)
    if _style_root_is_skipped(path, skip_paths):
        return {"status": 0, "output": "", "text": "", "truncated": False, "chars": 0, "omitted_chars": 0}
    out, err = StringIO(), StringIO()
    with redirect_stdout(out), redirect_stderr(err):
        status = _chkstyle_main(_style_check_argv(path, skip_folder_re, skip_path))
    output = "\n".join(chunk.rstrip() for chunk in (out.getvalue(), err.getvalue()) if chunk)
    capped = cap_text(output, max_output_chars=max_output_chars) if max_output_chars else {"text": output, "truncated": False, "chars": len(output), "omitted_chars": 0}
    return {"status": status, "output": output, **capped}

In [ ]:
argv = _style_check_argv(".")
assert "--skip-path" in argv
assert "_proc" in argv
assert "--skip-folder-re" not in argv
assert "--skip-path-re" in _style_check_argv(".", skip_folder_re=r"^_proc/")

In [ ]:
proc_report = run_style_check("_proc/01_read.ipynb")
assert proc_report["status"] == 0
assert proc_report["output"] == ""
assert notebook_validation_problems("_proc/01_read.ipynb") == []
assert not any(_style_root_is_skipped(problem["path"], skip_style_paths) for problem in _notebook_style_problems("."))

In [ ]:
#| export
def style_check(
    path: str = ".",
    skip_folder_re: str | None = None,  # Regex for folders to skip
    skip_path: str | None = None,  # Folder name/path to skip
    strict: bool = False,  # Raise when style hints or notebook hygiene problems are found
    delete_after_output: bool = False,  # Reset ~/.nbskill-errors.json after printing the global summary
    max_output_chars: int = 12000,  # Cap printed chkstyle output
    max_diagnostics: int = 200,  # Cap returned diagnostics
    fix: bool = False,  # Show conservative fix suggestions
    dry_run: bool = True,  # Keep fix mode non-mutating by default
    changed_only: bool = False,  # Report only diagnostics for changed code cells
    ref_a: str | None = "HEAD",  # First git ref for changed_only filtering
    ref_b: str | None = None,  # Second git ref; defaults to working tree
):
    "Print capped fast.ai style hints, notebook hygiene warnings, and global tool usage."
    chkstyle = run_style_check(path, skip_folder_re, skip_path, strict=False, max_output_chars=max_output_chars)
    report = style_report(
        path, chkstyle=chkstyle, max_output_chars=max_output_chars,
        max_diagnostics=max_diagnostics, changed_only=changed_only, ref_a=ref_a, ref_b=ref_b,
        skip_folder_re=skip_folder_re, skip_path=skip_path,
    )
    print(report["text"])
    if fix:
        print("\nFix suggestions:")
        fixes = report.get("fixes", [])
        if not fixes: print("- no deterministic fixes available")
        for item in fixes:
            mode = "would apply" if dry_run else "manual-review-required"
            print(f"- {mode}: {item['description']} ({item['path']})")
    if delete_after_output: reset_global_usage_summary()
    has_problems = bool(report["diagnostics"])
    if strict and (chkstyle["status"] or has_problems): raise RuntimeError(report["text"])
    return api_return(chkstyle["status"] or int(has_problems))

In [ ]:
#| export
def validate_nbs(
    path: str = "nbs",
    strict: bool = True,  # Raise when invalid metadata is found
):
    "Validate nbskill metadata needed for safe notebook tools."
    problems = notebook_validation_problems(path)
    if problems:
        print("Notebook validation errors:")
        for problem in problems:
            print(_format_problem(problem))
        if strict: raise ValueError("Notebook validation failed")
    else:
        print("Notebook validation: no invalid nbskill metadata found.")
    return api_return(int(bool(problems)))

In [ ]:
#| export
def code_source(cell):
    "Return source for code cells; ignore markdown and raw cells."
    return cell.source if cell.cell_type == "code" else None

In [ ]:
#| export
def _working_tree_code_sources(path):
    nb = read_nb(path)
    return {
        getattr(cell, "id", str(idx)): source
        for idx, cell in enumerate(nb.cells)
        if (source := code_source(cell)) is not None
    }

`style_check` is the direct Python wrapper around `style_report`. It captures fast.ai style output, appends notebook hygiene findings, prints the combined text report, and raises in strict mode when diagnostics are present.

In [ ]:
with write_tool_notebook("04_review_style_check.ipynb") as path:
    style_output = StringIO()
    with redirect_stdout(style_output): style_check(path, strict=False, max_output_chars=500, max_diagnostics=5)
    print("\n".join(style_output.getvalue().splitlines()[:6]))

In [ ]:
#| hide
with write_tool_notebook("04_review_style_fix.ipynb") as path:
    before = path.read_text(encoding="utf-8")
    output = StringIO()
    with redirect_stdout(output): style_check(str(path), fix=True)
    assert path.read_text(encoding="utf-8") == before

In [ ]:
with write_demo_notebook("04_review_validate_valid.ipynb") as valid_path:
    _write_review_notebook(valid_path, [mk_cell("assert True", cell_type="code")])
    assert notebook_validation_problems(valid_path) == []
    validate_nbs(str(valid_path), strict=False)

In [ ]:
with write_demo_notebook("04_review_validate_invalid.ipynb") as invalid_path:
    invalid_nb = _write_review_notebook(invalid_path, [
        mk_cell("x = 1", cell_type="code"),
        mk_cell("Some docs", cell_type="markdown"),
        mk_cell("assert True", cell_type="code"),
    ])
    invalid_nb.cells[1].metadata["nbskill"]["semantic_types"] = []
    invalid_nb.cells[2].metadata["nbskill"]["cell_type"] = "markdown"
    _write_nb(invalid_nb, invalid_path)
    codes = {problem["code"] for problem in notebook_validation_problems(invalid_path)}
    assert {"missing-cell-semantic-types", "cell-type-mismatch"} <= codes

In [ ]:
#| hide
with write_demo_notebook("04_review_invalid_python.ipynb") as syntax_path:
    syntax_nb = _write_review_notebook(syntax_path, [mk_cell("def broken(:", cell_type="code")])
    problem = next(problem for problem in notebook_validation_problems(syntax_path) if problem["code"] == "invalid-python-cell")
    assert problem["cell_id"] == syntax_nb.cells[0].id
    assert problem["line"] == 1

In [ ]:
with write_demo_notebook("04_review_export_hash.ipynb", base="nbs") as export_nb_path:
    export_nb = _write_review_notebook(export_nb_path, [mk_cell("#| default_exp sample_tool")])
    generated_py_path = exported_py_path(export_nb_path, export_nb)
    try:
        generated_py_path.parent.mkdir(parents=True, exist_ok=True)
        generated_py_path.write_text("actual = 1\n", encoding="utf-8")
        export_nb.metadata["nbskill"] = {"exported_py_hash": "bad"}
        _write_nb(export_nb, export_nb_path)
        codes = {problem["code"] for problem in notebook_validation_problems(export_nb_path)}
        assert "exported-py-hash-mismatch" in codes
    finally:
        if generated_py_path is not None: remove_demo_path(generated_py_path)

`notebook_validation_problems` is narrower than `style_report`: it checks nbskill metadata plus Python syntax in code cells, the prerequisites for reliable notebook tooling. `validate_nbs` exposes the same failures to Python callers.

In [ ]:
with write_demo_notebook("04_review_validation_example.ipynb") as validation_path:
    _write_review_notebook(validation_path, [
        mk_cell("x = 1", cell_type="code"),
        mk_cell("Some docs", cell_type="markdown"),
    ], stamp=False)
    problems = notebook_validation_problems(validation_path)
    print([problem["code"] for problem in problems[:4]])

### Code-cell diffs

Notebook diffs are noisy when metadata and outputs are included. `diff_nb` asks nbdev for code-cell source on each side of a comparison and prints only the added, changed, or deleted code blocks the caller requested.

In [ ]:
#| export
def _git_ref_path_error(path, ref):
    if ref is None: return None
    path = Path(path)
    root_cmd = subprocess.run(
        ["git", "-C", str(path.parent), "rev-parse", "--show-toplevel"],
        capture_output=True, text=True,
    )
    if root_cmd.returncode != 0:
        return (
            f"No git repository found for {str(path)!r}. "
            "For a temp/disposable notebook, call diff_nb(path, ref_a=None, ref_b=None) "
            "to review current code cells as additions."
        )
    root = Path(root_cmd.stdout.strip())
    try:
        rel = path.resolve().relative_to(root.resolve()).as_posix()
    except ValueError:
        return f"{str(path)!r} is outside git repository {str(root)!r}."
    spec = f"{ref}:{rel}"
    exists_cmd = subprocess.run(
        ["git", "-C", str(root), "cat-file", "-e", spec],
        capture_output=True, text=True,
    )
    if exists_cmd.returncode == 0: return None
    return (
        f"Could not find notebook {rel!r} at git ref {ref!r}. "
        "The notebook may be new relative to that ref, or the repository may not have a HEAD commit yet. "
        "Commit the notebook first, choose an existing ref/path, or pass ref_a=None and ref_b=None "
        "to review a disposable notebook as current working-tree additions."
    )

In [ ]:
#| export
def _git_root_rel(path):
    path = Path(path)
    root_cmd = subprocess.run(
        ["git", "-C", str(path.parent), "rev-parse", "--show-toplevel"],
        capture_output=True, text=True,
    )
    if root_cmd.returncode != 0: return None, None
    root = Path(root_cmd.stdout.strip())
    try: return root, path.resolve().relative_to(root.resolve()).as_posix()
    except ValueError: return None, None

In [ ]:
#| export
def _notebook_json_at_ref(path, ref):
    path = Path(path)
    if ref is None:
        return json.loads(path.read_text(encoding="utf-8"))
    root, rel = _git_root_rel(path)
    if root is None: return None
    show = subprocess.run(["git", "-C", str(root), "show", f"{ref}:{rel}"], capture_output=True, text=True)
    if show.returncode != 0: return None
    return json.loads(show.stdout)

In [ ]:
#| export
def _nbskill_metadata_by_cell(nb_json):
    cells = (nb_json or {}).get("cells", [])
    return {
        cell.get("id", str(idx)): (cell.get("metadata", {}) or {}).get("nbskill")
        for idx, cell in enumerate(cells)
    }

In [ ]:
#| export
def _nbskill_metadata_change_count(path, ref_a, ref_b):
    try:
        old = _nbskill_metadata_by_cell(_notebook_json_at_ref(path, ref_a))
        new = _nbskill_metadata_by_cell(_notebook_json_at_ref(path, ref_b))
    except (OSError, json.JSONDecodeError, TypeError):
        return 0
    keys = set(old) | set(new)
    return sum(1 for key in keys if old.get(key) != new.get(key) and (old.get(key) is not None or new.get(key) is not None))

In [ ]:
#| export
def _metadata_summary(count):
    if not count: return ""
    noun = "cell" if count == 1 else "cells"
    return f"Ignored nbskill metadata changes in {count} {noun}."

In [ ]:
#| export
def _changed_code_cell_ids(path, ref_a="HEAD", ref_b=None):
    ref_a, ref_b = none_if_string(ref_a), none_if_string(ref_b)
    try:
        old, new = nbs_pair(path, ref_a=ref_a, ref_b=ref_b, f=code_source)
    except Exception:
        return set()
    old = {cid: src for cid, src in old.items() if src is not None}
    new = {cid: src for cid, src in new.items() if src is not None}
    return {cid for cid in set(old) | set(new) if old.get(cid) != new.get(cid)}

In [ ]:
#| export
def _diff_filter_ids(path, ref_b=None, cell_id=None, after_id=None):
    if not cell_id and not after_id: return None
    nb_json = _notebook_json_at_ref(path, none_if_string(ref_b))
    cells = (nb_json or {}).get("cells", [])
    ids = [cell.get("id", str(idx)) for idx, cell in enumerate(cells)]
    selected = set(ids)
    if cell_id: selected &= set(str(cell_id).replace(",", " ").split())
    if after_id:
        if after_id not in ids: return selected & set()
        selected &= set(ids[ids.index(after_id) + 1:])
    return selected

In [ ]:
#| export
def _format_style_delta_report(path, diagnostics, changed_cell_ids, usage_text):
    header = f"Style delta for changed code cells in {path}: {len(diagnostics)} diagnostic(s) across {len(changed_cell_ids)} changed cell(s)"
    craft_text = _format_notebook_craft_report(diagnostics)
    if not diagnostics: return "\n\n".join([header, craft_text, usage_text])
    lines = [header]
    for item in diagnostics:
        cell = f" id={item['cell_id']}" if item.get("cell_id") else ""
        line = f" line={item['line']}" if item.get("line") else ""
        detail = item.get("detail") or item.get("code", "")
        lines.append(f"- {item.get('source', 'nbskill')}: {item.get('path', path)}{cell}{line} {detail}".rstrip())
    return "\n\n".join(["\n".join(lines), craft_text, usage_text])

### Public UI text surface

FastHTML and HTML-producing notebook cells often hide customer-visible copy inside Python component calls. A normal text search finds too much: headings, button labels, docstrings, route names, tests, exceptions, and implementation notes all look like strings. The public UI text surface gives agents a smaller first pass over likely visible copy before editing.

Use `visible_text_inventory` before an edit when the task is phrased in product or customer language. It answers: which notebook cell and symbol owns the visible words I may need to change? That reduces the manual `rg` loop where an agent has to compare notebook source, exported modules, and internal strings by hand.

Use `diff_nb(surface="public-ui")` after an edit when the review question is user-facing copy rather than Python structure. It keeps headings, labels, link text, image alt text, and similar UI literals separate from code churn, metadata churn, tests, and docstring edits.

This is deliberately heuristic. It is meant to support a faster notebook iteration loop, not to prove accessibility or render every possible runtime branch: inventory the likely visible text, make a focused edit, run the route/example smoke snippet, then review the public UI diff.

In [ ]:
#| export
_FASTHTML_VISIBLE_TEXT_CALLS = set("""
A Abbr Address Article Aside B Body Button Caption Code Dd Details Div Dt Em Fieldset
Figcaption Figure Footer Form H1 H2 H3 H4 H5 H6 Header Html I Img Input Label Legend Li Main Nav
Ol Option P Pre Section Select Small Span Strong Summary Table Tbody Td Textarea Tfoot Th
Thead Title Tr Ul Titled
""".split())
_VISIBLE_TEXT_KEYWORDS = {"alt", "aria_label", "aria-label", "label", "placeholder", "title"}
_NON_VISIBLE_TEXT_CALLS = {
    "AssertionError", "Exception", "FileNotFoundError", "OSError", "Path", "RuntimeError",
    "StringIO", "SystemExit", "TypeError", "ValueError", "Warning",
}

In [ ]:
#| export
def _literal_visible_text(node):
    if isinstance(node, ast.Constant) and isinstance(node.value, str): return node.value
    return None

In [ ]:
#| export
def _looks_like_visible_text(text):
    text = str(text or "").strip()
    if not text: return False
    if text.startswith(("#|", "http://", "https://")): return False
    return re.search(r"[A-Za-z0-9]", text) is not None

In [ ]:
#| export
def _source_top_level_symbols(source):
    try: tree = ast.parse(source)
    except SyntaxError: return []
    names = []
    for node in tree.body:
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef, ast.ClassDef)): names.append(node.name)
        elif isinstance(node, ast.Assign):
            names.extend(target.id for target in node.targets if isinstance(target, ast.Name))
    return names

In [ ]:
#| export
def _visible_text_records_from_source(path, cell_id, source):
    try: tree = ast.parse(source)
    except SyntaxError: return []
    symbols = ", ".join(_source_top_level_symbols(source)[:4])
    records = []
    for node in ast.walk(tree):
        if not isinstance(node, ast.Call): continue
        call = short_call_name(node.func, default="")
        visible_call = call in _FASTHTML_VISIBLE_TEXT_CALLS or (call[:1].isupper() and call not in _NON_VISIBLE_TEXT_CALLS)
        if not visible_call: continue
        for arg in node.args:
            text = _literal_visible_text(arg)
            if _looks_like_visible_text(text):
                records.append({
                    "path": str(path), "cell_id": str(cell_id), "line": getattr(arg, "lineno", None),
                    "symbol": symbols, "call": call, "text": text.strip(),
                })
        for kw in node.keywords:
            if kw.arg not in _VISIBLE_TEXT_KEYWORDS: continue
            text = _literal_visible_text(kw.value)
            if _looks_like_visible_text(text):
                records.append({
                    "path": str(path), "cell_id": str(cell_id), "line": getattr(kw.value, "lineno", None),
                    "symbol": symbols, "call": f"{call}.{kw.arg}", "text": text.strip(),
                })
    return records

In [ ]:
#| export
def _format_visible_text_records(records):
    if not records: return "No likely public UI text found"
    lines = ["cell_id | line | symbol | call | text"]
    for item in records:
        text = str(item.get("text", "")).replace("\n", "\\n")
        lines.append(
            f"{item.get('cell_id', '')} | {item.get('line') or ''} | "
            f"{item.get('symbol', '')} | {item.get('call', '')} | {text}"
        )
    return "\n".join(lines)

In [ ]:
#| export
def visible_text_inventory(path='.', include_re=None, max_records=200):
    "Print likely user-visible text from FastHTML/HTML-producing notebook code cells."
    pattern = re.compile(include_re) if include_re else None
    records = []
    for nb_path in notebook_paths(path):
        nb = read_nb(nb_path)
        for cell in nb.cells:
            source = code_source(cell)
            if source is None: continue
            for item in _visible_text_records_from_source(nb_path, getattr(cell, "id", ""), source):
                if pattern and not pattern.search(item["text"]): continue
                records.append(item)
                if len(records) >= max_records: break
            if len(records) >= max_records: break
        if len(records) >= max_records: break
    text = _format_visible_text_records(records)
    print(text)
    api_return(records)
    return records

In [ ]:
#| export
def _public_ui_diff_lines(path, cell_id, source):
    records = _visible_text_records_from_source(path, cell_id, source or "")
    return [f"{item['call']}: {item['text']}" for item in records]

In [ ]:
#| export
def _format_public_ui_diff(path, old, new, adds=True, changes=True, dels=False, selected=None):
    blocks = []
    cell_ids = list(dict.fromkeys([*old.keys(), *new.keys()]))
    for cid in cell_ids:
        if selected is not None and cid not in selected: continue
        old_lines = _public_ui_diff_lines(path, cid, old.get(cid, "")) if cid in old else []
        new_lines = _public_ui_diff_lines(path, cid, new.get(cid, "")) if cid in new else []
        if cid not in old:
            if not adds: continue
        elif cid not in new:
            if not dels: continue
        elif old_lines == new_lines or not changes: continue
        diff = source_diff("\n".join(old_lines), "\n".join(new_lines))
        if diff.strip(): blocks.append(f"--- public-ui cell {cid} ---\n{diff}")
    return "\n\n".join(blocks) if blocks else "No public UI text changes"

In [ ]:
#| export
def diff_nb(
    path: str,  # Notebook path
    ref_a: str|None = "HEAD",  # First git ref; use None for working tree
    ref_b: str|None = None,  # Second git ref; defaults to working tree
    adds: bool = True,  # Include code cells added in ref_b
    changes: bool = True,  # Include changed code cells
    dels: bool = False,  # Include deleted code cells
    cell_id: str | None = None,  # Limit output to comma/space-separated cell ids
    after_id: str | None = None,  # Limit output to cells after this id in ref_b/the working tree
    surface: str = "code",  # Review surface: "code" or "public-ui"
):
    "Print nbdev-style diffs for code cells, or a public UI text surface."
    ref_a, ref_b = none_if_string(ref_a), none_if_string(ref_b)
    surface = (surface or "code").replace("_", "-").lower()
    if surface not in {"code", "public-ui"}: raise ValueError("surface must be 'code' or 'public-ui'")
    if ref_a is None and ref_b is None and _git_root_rel(path)[0] is None:
        old, new = {}, _working_tree_code_sources(path)
    else:
        if msg := (_git_ref_path_error(path, ref_a) or _git_ref_path_error(path, ref_b)):
            api_error(msg)
        try: old, new = nbs_pair(path, ref_a=ref_a, ref_b=ref_b, f=code_source)
        except Exception as exc:
            detail = str(exc)
            hint = (
                f"Could not diff {path!r} against {ref_a!r}. "
                "The notebook may be new relative to that git ref, or the repository may not have a HEAD commit yet. "
                "Commit the notebook first, or pass ref_a=None and ref_b=None to review a disposable notebook as current working-tree additions.")
            if detail: hint += f"\nUnderlying error: {detail}"
            api_error(hint)
    old = {cid: src for cid, src in old.items() if src is not None}
    new = {cid: src for cid, src in new.items() if src is not None}
    selected = _diff_filter_ids(path, ref_b=ref_b, cell_id=cell_id, after_id=after_id)
    if surface == "public-ui":
        report = _format_public_ui_diff(path, old, new, adds=adds, changes=changes, dels=dels, selected=selected)
        metadata_summary = _metadata_summary(_nbskill_metadata_change_count(path, ref_a, ref_b))
        if metadata_summary: report = f"{report}\n\n{metadata_summary}"
        print(report)
        api_return(report)
        return report
    blocks = []
    if adds:    blocks += [(cid, source_diff("", new[cid])) for cid in new if cid not in old]
    if changes: blocks += [(cid, source_diff(old[cid], new[cid])) for cid in new if cid in old and new[cid] != old[cid]]
    if dels:    blocks += [(cid, source_diff(old[cid], "")) for cid in old if cid not in new]
    if selected is not None: blocks = [(cid, diff) for cid, diff in blocks if cid in selected]
    text = "\n\n".join(f"--- code cell {cid} ---\n{diff}" for cid, diff in blocks if diff.strip())
    metadata_summary = _metadata_summary(_nbskill_metadata_change_count(path, ref_a, ref_b))
    if text and metadata_summary: report = f"{text}\n\n{metadata_summary}"
    elif text: report = text
    elif metadata_summary: report = f"No code cell changes\n{metadata_summary}"
    else: report = "No code cell changes"
    print(report)
    api_return(report)
    return report

In [ ]:
#| hide
from tempfile import gettempdir

with write_tool_notebook("04_review_fixture_diff.ipynb", base=gettempdir()) as path:
    out = StringIO()
    with redirect_stdout(out): diff_nb(str(path), ref_a=None, ref_b=None)
    test_eq("double_answer" in out.getvalue(), True)

In [ ]:
from tempfile import gettempdir

with write_demo_notebook("04_review_no_git.ipynb", base=gettempdir()) as path:
    _write_review_notebook(path, [mk_cell("x = 1", cell_type="code")])
    try:
        diff_nb(str(path))
    except (SystemExit, ValueError) as exc:
        if isinstance(exc, SystemExit): assert exc.code == 1
        else:
            msg = str(exc)
            assert "disposable notebook" in msg and "ref_a=None" in msg and "ref_b=None" in msg
    out = StringIO()
    with redirect_stdout(out):
        diff_nb(str(path), ref_a=None, ref_b=None)
    report = out.getvalue()
    assert "--- code cell" in report and "+x = 1" in report

In [ ]:
def _committed_review_notebook(root, cells=None):
    root.mkdir()
    path = root / "demo.ipynb"
    _write_review_notebook(path, cells or [mk_cell("x = 1", cell_type="code")])
    subprocess.run(["git", "init"], cwd=root, check=True, capture_output=True)
    subprocess.run(["git", "add", "demo.ipynb"], cwd=root, check=True, capture_output=True)
    subprocess.run([
        "git", "-c", "user.name=Nbskill", "-c", "user.email=nbskill@example.com",
        "commit", "-m", "base",
    ], cwd=root, check=True, capture_output=True)
    return path

In [ ]:
root = demo_path("04_review_git_metadata")
try:
    path = _committed_review_notebook(root)
    nb = read_nb(path)
    nb.cells[0].metadata["nbskill"] = {"cell_type": "code", "semantic_types": []}
    _write_nb(nb, path)
    out = StringIO()
    with redirect_stdout(out):
        diff_nb(str(path))
    text = out.getvalue()
    assert "No code cell changes" in text
    assert "Ignored nbskill metadata changes in 1 cell" in text
finally:
    remove_demo_path(root)

In [ ]:
root = demo_path("04_review_git_filters")
try:
    path = _committed_review_notebook(root)
    nb = read_nb(path)
    first_id = nb.cells[0].id
    nb.cells[0].source = "x = 2"
    nb.cells.append(mk_cell("y = 3", cell_type="code"))
    _write_nb(nb, path)
    out = StringIO()
    with redirect_stdout(out): diff_nb(str(path), cell_id=first_id)
    text = out.getvalue()
    assert "x = 2" in text and "y = 3" not in text
    out = StringIO()
    with redirect_stdout(out): diff_nb(str(path), after_id=first_id)
    text = out.getvalue()
    assert "y = 3" in text and "x = 2" not in text
    delta = style_report(str(path), changed_only=True)
    assert delta["summary"]["changed_only"] is True
    assert delta["summary"]["changed_cell_count"] == 2
finally:
    remove_demo_path(root)

In [ ]:
#| hide
root = demo_path("04_review_advice_changed_only")
try:
    path = _committed_review_notebook(root, [
        mk_cell("#| default_exp demo"),
        mk_cell("#| export\ndef parse_value(text):\n    return text.strip().lower()"),
    ])
    nb = read_nb(path)
    nb.cells.append(mk_cell("#| export\ndef clean_value(value):\n    return value.strip().lower()"))
    changed_id = nb.cells[-1].id
    _write_nb(nb, path)

    report = style_report(str(path), changed_only=True)
    assert report["summary"]["changed_only"] is True
    assert report["summary"]["craft_problem_count"] >= 1
    assert "Notebook craft:" in report["text"]
    assert any(
        item["code"] == "similar-function" and item.get("cell_id") == changed_id
        for item in report["diagnostics"]
    )
    craft = _craft_diagnostics(report["diagnostics"])
    assert craft
    assert all(item.get("cell_id") == changed_id for item in craft)
    assert all(
        item.get("cell_id") == changed_id
        for item in report["diagnostics"]
        if item.get("code") in {"similar-function", "misplaced-function", "private-boundary-reuse"}
    )
finally:
    remove_demo_path(root)

In [ ]:
def _write_style_problem_notebook(path):
    long_source = "\n".join([f"x{i} = {i}" for i in range(31)])
    long_markdown = "\n".join([f"paragraph {i}" for i in range(21)])
    mixed_import = mk_cell("import json", cell_type="code")
    mixed_import.outputs = [dict(output_type="stream", name="stdout", text="json\n")]
    _write_review_notebook(path, [
        mk_cell(
            "#| export\n"
            "def a():\n"
            "    pass\n"
            "def b():\n"
            "    \"\"\"Describe b.\"\"\"\n"
            "    pass\n"
            "def c():\n"
            "    \"\"\"Line one.\n"
            "    Line two.\"\"\"\n"
            "    pass\n"
            "def _private():\n"
            "    pass",
            cell_type="code",
        ),
        mk_cell(long_source, cell_type="code"),
        mk_cell(long_markdown, cell_type="markdown"),
        mk_cell("assert 1 == 1\nassert 2 == 2\nassert 3 == 3\nassert 4 == 4", cell_type="code"),
        mk_cell("#| export\nimport os", cell_type="code"),
        mk_cell("#| export\nimport os", cell_type="code"),
        mixed_import,
        mk_cell("import sys", cell_type="code"),
        mk_cell("import sys", cell_type="code"),
        mk_cell("result = later_helper()", cell_type="code"),
        mk_cell("def later_helper():\n    return 1", cell_type="code"),
        mk_cell("def loader():\n    return MissingPath('x')", cell_type="code"),
    ])

`_write_literacy_problem_notebook` builds a compact demo notebook for the literate review contract. The fixture keeps compliant, missing-docs, missing-example, missing-test, and multiline-docstring cases in one place so each warning code has a precise example.

In [ ]:
def _write_literacy_problem_notebook(path):
    _write_review_notebook(path, [
        mk_cell("`ready` has docs, a docstring, an example, and a test.", cell_type="markdown"),
        mk_cell("#| export\ndef ready(x):\n    \"Return x plus one.\"\n    return x + 1", cell_type="code"),
        mk_cell("`no_example` has docs and a test but no example call.", cell_type="markdown"),
        mk_cell("#| export\ndef no_example(x):\n    \"Return x plus one.\"\n    return x + 1", cell_type="code"),
        mk_cell("`no_test` has docs and an example but no test assertion.", cell_type="markdown"),
        mk_cell("#| export\ndef no_test(x):\n    \"Return x plus one.\"\n    return x + 1", cell_type="code"),
        mk_cell("`bad_doc` has a Markdown note but its docstring is too long.", cell_type="markdown"),
        mk_cell("#| export\ndef bad_doc(x):\n    \"\"\"Return x plus one.\n    Keep this second line out of file_context.\"\"\"\n    return x + 1", cell_type="code"),
        mk_cell("`eq_checked`, `ne_checked`, and `fail_checked` cover fastcore test helpers.", cell_type="markdown"),
        mk_cell("#| export\ndef eq_checked(x):\n    \"Return x plus one.\"\n    return x + 1", cell_type="code"),
        mk_cell("#| export\ndef ne_checked(x):\n    \"Return x plus one.\"\n    return x + 1", cell_type="code"),
        mk_cell("#| export\ndef fail_checked():\n    \"Raise a demo failure.\"\n    raise ValueError(\"boom\")", cell_type="code"),
        mk_cell("#| export\ndef no_docs(x):\n    \"Return x plus one.\"\n    return x + 1", cell_type="code"),
        mk_cell("#| export\ndef _hidden(x):\n    return x\n\ndef test_helper(x):\n    return x", cell_type="code"),
        mk_cell("ready(1)\nno_docs(1)\nno_test(1)\nbad_doc(1)\neq_checked(1)\nne_checked(1)\ntry:\n    fail_checked()\nexcept ValueError:\n    pass", cell_type="code"),
        mk_cell("assert ready(1) == 2\nassert no_docs(1) == 2\nassert no_example(1) == 2\nassert bad_doc(1) == 2\ntest_eq(eq_checked(1), 2)\ntest_ne(ne_checked(1), 3)\ntest_fail(lambda: fail_checked())\nassert _hidden(1) == 1\nassert test_helper(1) == 1", cell_type="code"),
    ])

In [ ]:
with write_demo_notebook("04_review_literacy.ipynb") as path:
    _write_literacy_problem_notebook(path)
    problems = public_function_literacy_problems(path)
    codes = {(problem.get("symbol"), problem["code"]) for problem in problems}
    missing_codes = {("no_docs", "public-function-markdown"), ("no_example", "public-function-example"), ("no_test", "public-function-test")}
    ready_codes = {code for symbol, code in codes if symbol == "ready"}
    assert missing_codes <= codes
    assert not ready_codes


In [ ]:
with write_demo_notebook("04_review_literacy_docstrings.ipynb") as path:
    _write_literacy_problem_notebook(path)
    problems = public_function_literacy_problems(path)
    codes = {(problem.get("symbol"), problem["code"]) for problem in problems}
    tested_codes = {
        ("eq_checked", "public-function-test"),
        ("ne_checked", "public-function-test"),
        ("fail_checked", "public-function-test"),
    }
    assert ("bad_doc", "public-function-docstring") in codes
    assert not tested_codes & codes
    assert not any(problem.get("symbol") in {"_hidden", "test_helper"} for problem in problems)

In [ ]:
#| hide
with write_demo_notebook("04_review_literacy_documentation.ipynb") as path:
    _write_review_notebook(path, [
        mk_cell("`mentioned_only`.", cell_type="markdown"),
        mk_cell("#| export\ndef mentioned_only():\n    \"Return a value.\"\n    return 1", cell_type="code"),
    ])
    codes = {(problem.get("symbol"), problem["code"]) for problem in public_function_literacy_problems(path)}
    assert ("mentioned_only", "public-function-markdown") in codes

In [ ]:
#| hide
with write_demo_notebook("04_review_literacy_chart.ipynb") as path:
    _write_literacy_problem_notebook(path)
    report = style_report(path)
    chart = report["problem_chart"]["by_code"]
    expected_chart = {
        "public-function-markdown": 1,
        "public-function-example": 1,
        "public-function-test": 1,
        "public-function-docstring": 1,
    }
    assert all(chart[code] == count for code, count in expected_chart.items())
    assert report["summary"]["craft_problem_count"] >= 4
    assert report["summary"]["public_function_story_problem_count"] == 3
    assert "Notebook craft:" in report["text"]
    assert "add a short Markdown rationale cell" in report["text"]

In [ ]:
#| hide
with write_demo_notebook("04_review_style.ipynb") as path:
    _write_style_problem_notebook(path)
    report = _format_notebook_style_report(path)
    assert "Notebook craft:" in report
    assert "large-cell" in report
    assert "split this into markdown rationale" in report
    assert "markdown content lines" in report
    assert "multi-problem-test" in report
    assert "multicell" in report
    assert "multiple-semantic-types" in report
    assert "scope=exported" in report
    assert "scope=internal" in report
    assert "cell-order" in report
    assert "missing-import" in report
    assert "public-function-docstring" in report
    assert "symbol='a'" in report
    assert "symbol='c'" in report
    assert "symbol='b'" in report
    assert "symbol='_private'" not in report

In [ ]:
print(report)

Notebook style report:
- missing-cell-nbskill-metadata: nbs/data/04_review_style.ipynb id=18cf6a7f
- missing-cell-nbskill-metadata: nbs/data/04_review_style.ipynb id=3e6417a6
- missing-cell-nbskill-metadata: nbs/data/04_review_style.ipynb id=9d964403
- missing-cell-nbskill-metadata: nbs/data/04_review_style.ipynb id=56e0e6c4
- missing-cell-nbskill-metadata: nbs/data/04_review_style.ipynb id=4debe573
- missing-cell-nbskill-metadata: nbs/data/04_review_style.ipynb id=742a5cf5
- missing-cell-nbskill-metadata: nbs/data/04_review_style.ipynb id=f41bdaef
- missing-cell-nbskill-metadata: nbs/data/04_review_style.ipynb id=3b1110e1
- missing-cell-nbskill-metadata: nbs/data/04_review_style.ipynb id=23f2d90a
- missing-cell-nbskill-metadata: nbs/data/04_review_style.ipynb id=4268cb78
- missing-cell-nbskill-metadata: nbs/data/04_review_style.ipynb id=762d5962
- missing-cell-nbskill-metadata: nbs/data/04_review_style.ipynb id=711b360d
- large-cell: nbs/data/04_review_style.ipynb id=18cf6a7f function

In [ ]:
with write_demo_notebook("04_review_generated.ipynb") as generated_path:
    generated_nb = _write_review_notebook(generated_path, [mk_cell("#| default_exp big_review_demo")])
    generated_py_path = exported_py_path(generated_path, generated_nb)
    try:
        generated_py_path.parent.mkdir(parents=True, exist_ok=True)
        generated_py_path.write_text("\n".join(f"x{i} = {i}" for i in range(1001)), encoding="utf-8")
        size_codes = {problem["code"] for problem in notebook_size_problems(generated_path)}
        assert "large-generated-py" in size_codes
    finally:
        if generated_py_path is not None: remove_demo_path(generated_py_path)

In [ ]:
with write_demo_notebook("04_review_style.ipynb") as path:
    _write_style_problem_notebook(path)
    custom_map = demo_path("04_review_errors.json")
    old_map = os.environ.get("NBSKILL_FAILURE_MAP")
    try:
        os.environ["NBSKILL_FAILURE_MAP"] = str(custom_map)
        with redirect_stdout(StringIO()):
            style_check(str(path), delete_after_output=True)
        assert "usage:" in _format_global_usage_summary()
        assert not custom_map.exists()
    finally:
        if old_map is None: os.environ.pop("NBSKILL_FAILURE_MAP", None)
        else: os.environ["NBSKILL_FAILURE_MAP"] = old_map

In [ ]:
with write_demo_notebook("04_review_style.ipynb") as path:
    _write_style_problem_notebook(path)
    try:
        with redirect_stdout(StringIO()): style_check(str(path), strict=True)
    except RuntimeError:
        pass
    else:
        raise AssertionError("strict style_check should raise for notebook hygiene problems")

In [ ]:
assert code_source(mk_cell("plain docs", cell_type="markdown")) is None
assert code_source(mk_cell("answer = 42", cell_type="code")) == "answer = 42"

`diff_nb` deliberately compares code-cell source only. Markdown edits and notebook metadata churn stay out of the main diff so reviewers can see the executable behavior that changed.

In [ ]:
#| eval: false
#| eval: false
diff_root = demo_path("04_review_diff_example")
remove_demo_path(diff_root)
try:
    diff_root.mkdir()
    diff_path = diff_root / "demo.ipynb"
    _write_nb(new_nb([
        mk_cell("value = 1\nvalue", cell_type="code"),
        mk_cell("Original note", cell_type="markdown"),
    ]), diff_path)
    subprocess.run(["git", "init"], cwd=diff_root, check=True, capture_output=True)
    subprocess.run(["git", "add", "demo.ipynb"], cwd=diff_root, check=True, capture_output=True)
    subprocess.run([
        "git", "-c", "user.name=Nbskill", "-c", "user.email=nbskill@example.com",
        "commit", "-m", "base",
    ], cwd=diff_root, check=True, capture_output=True)

    diff_nb_json = read_nb(diff_path)
    diff_nb_json.cells[0].source = "value = 2\nvalue"
    diff_nb_json.cells[1].source = "Updated note that will not appear in the code diff"
    _write_nb(diff_nb_json, diff_path)

    diff_output = StringIO()
    with redirect_stdout(diff_output):
        diff_nb(str(diff_path), ref_a="HEAD")
    print(diff_output.getvalue())
finally:
    remove_demo_path(diff_root)

In [ ]:
#| hide
from tempfile import gettempdir

ui_source = """
def _hero():
    return Div(H1('Welcome'), Button('Start now'), Img(src='/logo.png', alt='Company logo'))
raise ValueError('internal diagnostic')
"""
records = _visible_text_records_from_source("demo.ipynb", "ui-cell", ui_source)
texts = {item["text"] for item in records}
assert {"Welcome", "Start now", "Company logo"} <= texts
assert "internal diagnostic" not in texts

with write_demo_notebook("04_review_ui_inventory.ipynb", base=gettempdir()) as ui_path:
    _write_review_notebook(ui_path, [mk_cell(ui_source, cell_type="code")])
    out = StringIO()
    with redirect_stdout(out):
        inventory = visible_text_inventory(ui_path, include_re="Welcome|logo")
    assert [item["text"] for item in inventory] == ["Welcome", "Company logo"]

    out = StringIO()
    with redirect_stdout(out):
        report = diff_nb(str(ui_path), ref_a=None, ref_b=None, surface="public-ui")
    public_ui_text = out.getvalue()
    assert report == public_ui_text.rstrip()
    assert "--- public-ui cell" in public_ui_text
    assert "+H1: Welcome" in public_ui_text
    assert "+Button: Start now" in public_ui_text
    assert "+Img.alt: Company logo" in public_ui_text
    assert "internal diagnostic" not in public_ui_text

Here is the intended workflow on a tiny FastHTML-shaped notebook: first inventory likely visible text, then review just the public UI surface. The output is compact enough for an agent to paste into a status update or use as a pre-edit checklist.

In [ ]:
from tempfile import gettempdir

ui_source = """
def _hero():
    return Div(
        H1('Welcome'),
        P('Compare providers in minutes.'),
        Button('Start now'),
        Img(src='/logo.png', alt='Company logo'),
    )

raise ValueError('internal diagnostic, not customer copy')
"""

with write_demo_notebook("04_review_public_ui_demo.ipynb", base=gettempdir()) as ui_path:
    _write_review_notebook(ui_path, [mk_cell(ui_source, cell_type="code")])

    print("Inventory before editing:")
    visible_text_inventory(ui_path)

    print("\nPublic UI diff for the current notebook contents:")
    diff_nb(str(ui_path), ref_a=None, ref_b=None, surface="public-ui")

Review functions are direct Python APIs; MCP consolidates their common diagnostics through `doctor`.